# TSFM tool workflow: onboard an asset, forecast it, watch it for anomalies

A single end-to-end walkthrough of the TSFM server's tools, in the order an agent would actually
call them to stand up monitoring for a **newly commissioned chiller** - from "what do I even have?"
to a forecasting model promoted in the catalog with its decision recorded.

**Everything here runs.** It uses classical sktime estimators (Naive, Theta, AutoREG, SubLOF), so it
is a real fit on every cell - no torch, no HuggingFace, no network. That is the point: it exercises
the tools for real, not with mocks.

The workflow touches ~26 tools across seven stages:

1. **Orient** - what tasks, models, features, domains exist
2. **Understand the data** - profile, characterize, quality-check the series
3. **Discover models** - search / filter / shortlist / preflight
4. **Engineer features** - rank and extract, self-supervised
5. **Forecast** - a bake-off through one `run_recipe`, pick a winner
6. **Detect anomalies** - the same `run_recipe`, a detector card
7. **Author + ledger** - register the winner, version it, read the run/result history

## Setup — no torch, in-memory store so it is self-contained

In [ ]:
import os, sys, json, warnings
warnings.filterwarnings("ignore")
SRC = os.path.abspath("src")
os.environ["PYTHONPATH"] = SRC + os.pathsep + os.environ.get("PYTHONPATH", "")
os.environ["TSFM_STORE"] = "memory"
sys.path.insert(0, SRC)

import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from servers.tsfm import main as M
from servers.tsfm.io import refs
from servers.tsfm.reasoning import feature_selection as FS

def show(result, *keys):
    """Tool results are pydantic models; dump to a dict and optionally pick keys."""
    d = result.model_dump() if hasattr(result, "model_dump") else result
    return {k: d.get(k) for k in keys} if keys else d

print("store:", type(M._STORE).__name__)

## The asset: Chiller 7, two weeks of hourly load

Daily cycle, a weekly component, slow drift, noise - and **three injected fault spikes** at hours
90, 200, 300 so the anomaly stage has ground truth to score against.

In [ ]:
SP, N = 24, 336                      # hourly, 14 days
TRUTH, TOL = [90, 200, 300], 2
rng = np.random.RandomState(1)
t = np.arange(N)
load = (22 + 6*np.sin(t/SP*2*np.pi) + 2*np.sin(t/168*2*np.pi) + 0.01*t
        + rng.normal(0, .4, N))
for s in TRUTH:
    load[s] += 11.0
ref = refs.materialize_iot(load, asset_id="chiller_7")

fig, ax = plt.subplots(figsize=(12, 3))
ax.plot(t, load, lw=.8)
for s in TRUTH: ax.axvline(s, color="red", alpha=.3)
ax.set(title="Chiller 7 load — 14 days hourly (injected faults in red)", xlabel="hour")
plt.tight_layout(); plt.show()
print("file pointer:", ref)

## Stage 1 — Orient

Before touching the data, an agent asks what the server offers: the task contracts, and how big the
model and feature catalogs are.

In [ ]:
tasks = show(M.list_tasks())["tasks"]
print("tasks available:", [t.get("task_id") or t.get("id") or t for t in tasks][:8])
print("models in catalog :", show(M.count_models()))
print("features in catalog:", show(M.count_features()))
print("domains            :", show(M.list_domains())["domains"])

The catalogs read 0 here because the in-memory store starts empty (over CouchDB they load the seeded
cards). That is fine - this notebook **builds** its own catalog as it goes, which is exactly what an
agent onboarding a new asset does.

## Stage 2 — Understand the data

Three read tools turn the raw series into structured evidence: `profile_series` (fast facts),
`characterize_series` (pattern evidence, written to a file pointer), and `data_quality` (NaN check +
a cleaned pointer for downstream use).

In [ ]:
prof = show(M.profile_series(dataset_path=ref, timestamp_column="timestamp"))
print("dominant_period    :", prof["dominant_period"], "(expect ~24)")
print("seasonality_strength:", round(prof["seasonality_strength"], 3))
print("trend_strength     :", round(prof["trend_strength"], 3), "| non_stationary:", prof["non_stationary"])
print("n_missing          :", prof["n_missing"], "| value_range:", [round(v,1) for v in prof["value_range"]])

ch = show(M.characterize_series(dataset_path=ref, timestamp_column="timestamp"))
print("\ncharacterize summary:", str(ch["summary"])[:110])
print("evidence written to :", ch["evidence_file"])

dq = show(M.data_quality(dataset_path=ref, timestamp_column="timestamp"))
print("\ndata_quality: rows_in", dq["rows_in"], "-> rows_out", dq["rows_out"], "| cleaned:", dq["cleaned_file"])

`profile_series` recovered the 24-hour period straight from the data - the agent now knows the
seasonal period to pass to seasonal models, without being told.

## Stage 3 — Engineer features (self-supervised, no labels)

`select_features` ranks a candidate set of extractors by how well each predicts the next value, and
returns the shortlist worth keeping. `extract_features` then computes them. This is the feature half
of the workflow, fully inspectable.

In [ ]:
candidates = list(FS.EXTRACTORS)[:12]                       # a candidate pool from the 228-strong library
print("candidate extractors:", candidates)

sel = show(M.select_features(dataset_path=ref, channel="value", extractors=candidates))
print("\nshortlist kept      :", sel["selected"])
print("scored against      :", sel["reference"], "| lookback:", sel["lookback"])

vals = show(M.extract_features(dataset_path=ref,
                               extractors=sel["selected"] or candidates[:3],
                               target_columns=["value"], window=48))
print("\nextract_features    :", vals["message"])
print("columns             :", vals["columns"])
print("first window        :", [round(v,3) for v in vals["features"][0]])

An **empty shortlist is a valid result**, not a bug: on a smooth, strongly seasonal series none of
the candidate extractors beat the `mean` baseline by the `cd_margin`, so `select_features` keeps
none. The extract step below falls back to the candidate head so the workflow continues - exactly
how an agent would handle "nothing cleared the bar." On a messier multivariate signal the shortlist
is typically non-empty.

## Stage 4 — Discover models for the task

Register a small stable of classical forecasters, then use the discovery tools an agent would use to
find and vet them: `search_models`, `find_models`, `describe_candidates`, `describe_models`, and
`resolve_model` (the preflight that confirms a card can actually load).

In [ ]:
STABLE = {
    "naive_last":     ("sktime.forecasting.naive.NaiveForecaster", {"strategy": "last"}),
    "naive_seasonal": ("sktime.forecasting.naive.NaiveForecaster", {"strategy": "last", "sp": SP}),
    "theta":          ("sktime.forecasting.theta.ThetaForecaster", {"sp": SP}),
    "autoreg":        ("sktime.forecasting.auto_reg.AutoREG", {"lags": SP}),
}
for mid, (cls, params) in STABLE.items():
    M.register_model({"model_id": mid, "description": f"{mid} forecaster for chiller-7 onboarding",
                      "task_ids": ["tsfm_forecasting"], "provenance": "trained", "domain": "energy",
                      "sktime_class": cls, "params": params, "tags": ["classical", "forecast"]})
print("registered:", list(STABLE), "| catalog now:", show(M.count_models())["total"], "models")

print("\nsearch_models('seasonal'):",
      [m["model_id"] for m in show(M.search_models(text="seasonal"))["models"]])
cands = show(M.describe_candidates(task_id="tsfm_forecasting", top_k=4))["candidates"]
print("describe_candidates     :", [c["model_id"] for c in cands])

print("\nresolve_model preflight:")
for mid in STABLE:
    r = show(M.resolve_model(mid))
    print(f"   {mid:16s} resolvable={r['resolvable']}  regime={r['training_regime']}")

## Stage 5 — Forecasting bake-off through one `run_recipe`

Every model goes through the **same** `run_recipe` call - only the `model_id` changes. That is what
makes the comparison fair. Score is a rolling-origin backtest (MAPE); `folds` tells you how many
windows it averaged, so the numbers are comparable.

In [ ]:
FH = [1, 2, 3, 4, 5, 6]
board = {}
for mid in STABLE:
    r = show(M.run_recipe(dataset_path=ref, timestamp_column="timestamp", target_columns=["value"],
                          asset_id="chiller_7",
                          recipe={"estimator": {"model_id": mid}, "fh": FH,
                                  "eval": {"metrics": ["mape"]}}))
    if "error" not in r:
        board[mid] = {"MAPE": r["backtest_score"], "folds": r["folds"], "regime": r["training_regime"]}
        print(f"  {mid:16s} MAPE={r['backtest_score']:.4f}  folds={r['folds']}")
    else:
        print(f"  {mid:16s} ERROR: {r['error'][:60]}")

tbl = pd.DataFrame(board).T.sort_values("MAPE")
display(tbl)
winner = tbl.index[0]
print("winner:", winner, "@ MAPE", round(tbl.loc[winner, "MAPE"], 4))

## Stage 6 — Anomaly detection, same `run_recipe`

Switch `task` to `tsfm_anomaly_detection` and hand it a detector card. It fits, predicts dense
labels, and writes them to a results file. Because we injected the faults, we can score precision and
recall.

In [ ]:
M.register_model({"model_id": "sublof", "description": "Subsequence-LOF density detector for chiller-7",
                  "task_ids": ["tsfm_anomaly_detection"], "provenance": "trained", "domain": "energy",
                  "sktime_class": "sktime.detection.lof.SubLOF",
                  "params": {"window_size": SP, "n_neighbors": 5, "novelty": True}})

ad = show(M.run_recipe(dataset_path=ref, timestamp_column="timestamp", target_columns=["value"],
                       asset_id="chiller_7",
                       recipe={"task": "tsfm_anomaly_detection", "estimator": {"model_id": "sublof"}}))
rec = json.loads(open(ad["results_file"][7:]).read())
flagged = rec.get("anomaly_indices", [])
tp = sum(1 for s in TRUTH if any(abs(f-s) <= TOL for f in flagged))
fp = sum(1 for f in flagged if all(abs(f-s) > TOL for s in TRUTH))
prec = tp/(tp+fp) if (tp+fp) else 0.0; recall = tp/len(TRUTH)
print(f"flagged {len(flagged)} points: {flagged}")
print(f"ground truth {TRUTH}  ->  precision={prec:.2f}  recall={recall:.2f}")

fig, ax = plt.subplots(figsize=(12, 3))
ax.plot(t, load, lw=.7)
ax.scatter(flagged, load[flagged], color="red", zorder=5, label="flagged")
ax.set(title=f"SubLOF anomalies (precision {prec:.2f}, recall {recall:.2f})"); ax.legend()
plt.tight_layout(); plt.show()

SubLOF caught two of the three faults at **precision 1.00** but missed the one at hour 90 (recall
0.67) - a real, imperfect result, not staged. The spike at 90 lands early, before the detector has a
full window of context to call it unusual. That is the kind of honest signal these tools are meant
to surface; the fix would be a shorter `window_size` or a second detector, which the bake-off
pattern from stage 5 handles directly.

## Stage 7 — Author the decision into the catalog, then read the ledger

Promote the winner, retire the losers with their scores, cut a new version, and inspect lineage -
then read the full run and result history back out. This is what makes the workflow reproducible:
the decision lives in the catalog and the ledger, not just in this notebook.

In [ ]:
# promote the winner
M.update_model(winner, {"tags": ["classical", "forecast", "recommended"],
                        "description": f"SELECTED for chiller-7 6h forecasting: MAPE "
                                       f"{board[winner]['MAPE']:.4f} over {board[winner]['folds']} folds."})
# retire the losers with the reason recorded
for mid in board:
    if mid != winner:
        M.deprecate_model(mid, reason=f"lost chiller-7 bake-off: MAPE {board[mid]['MAPE']:.4f} "
                                      f"vs {board[winner]['MAPE']:.4f}")
# version the winner
M.new_model_version(winner, {"description": f"{winner} v2 — retuned for chiller-7"},
                    new_model_id=f"{winner}_v2")
lin = show(M.get_model_lineage(f"{winner}_v2"))
print("lineage of", f"{winner}_v2", "-> supersedes:", lin.get("supersedes") or lin.get("ancestors"))

live = [m["model_id"] for m in show(M.find_models(task_id="tsfm_forecasting"))["models"]]
print("active forecasting models now:", sorted(live))

In [ ]:
# the ledger: every run and its persisted result
runs = show(M.list_runs())["runs"]
print(f"list_runs: {len(runs)} executions recorded")
for r in runs[:6]:
    print(f"   {r.get('run_id'):18s} regime={str(r.get('training_regime')):13s} score={r.get('backtest_score')}")

fr = show(M.list_results(task_type="tsfm_forecasting", asset_id="chiller_7"))["results"]
print(f"\nlist_results (forecasting): {len(fr)} results")
if fr:
    one = show(M.get_result(task_type="tsfm_forecasting", result_id=fr[0]["result_id"]))
    print("get_result ->", {k: one.get(k) for k in ("result_id", "results_file", "summary")})

## What this exercised

In one runnable pass, on real fits with no torch:

| stage | tools |
|---|---|
| orient | `list_tasks`, `count_models`, `count_features`, `list_domains` |
| understand data | `profile_series`, `characterize_series`, `data_quality` |
| features | `select_features`, `extract_features` |
| discover | `register_model`, `search_models`, `describe_candidates`, `resolve_model` |
| forecast | `recipe_template`*, `run_recipe` (x4), one fair bake-off |
| anomaly | `run_recipe` (detector task) |
| author + ledger | `update_model`, `deprecate_model`, `new_model_version`, `get_model_lineage`, `find_models`, `list_runs`, `list_results`, `get_result` |

The throughline is the **catalog + recipe + ledger** design: models are pointer cards, one
`run_recipe` serves every task, and every execution is both returned as a `results_file` and made
findable later by `list_runs` / `get_result`. Swap `TSFM_STORE=memory` for CouchDB and the same
notebook runs against the seeded catalog with durable runs; swap a classical card for a TTM card and
the same `run_recipe` calls a foundation model.